<a href="https://colab.research.google.com/github/YoverOlivares/Sistema_de_riego_inteligente/blob/main/Entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sistema de Riego Inteligente Predictivo con
Aprendizaje Local: Un Proyecto IoT Offline para Agricultura
Sostenible

In [1]:
import pandas as pd
import random

# 1. Configuración
CANTIDAD_DATOS = 20000
suelo_list = []
temp_list = []
hum_aire_list = []
tiempo_riego_list = []

print(f"Generando {CANTIDAD_DATOS} datos de entrenamiento...")

# 2. Generación
for _ in range(CANTIDAD_DATOS):
    # Entradas aleatorias realistas
    suelo = round(random.uniform(0, 100), 2)
    temp = round(random.uniform(10, 32), 2)
    hum_aire = round(random.uniform(20, 95), 2)

    # Lógica "Maestra" (Imita tu código C++ actual)
    tiempo = 0.0
    if suelo < 40.0:
        tiempo = 5.0
        if temp > 25.0:
            tiempo += (temp - 25.0) * 0.5
        if tiempo > 15.0: tiempo = 15.0

    # Ruido (Variación humana)
    if tiempo > 0:
        tiempo += random.uniform(-0.1, 0.1)
        tiempo = max(0, round(tiempo, 2))

    suelo_list.append(suelo)
    temp_list.append(temp)
    hum_aire_list.append(hum_aire)
    tiempo_riego_list.append(tiempo)

# 3. Guardado
df = pd.DataFrame({
    'HumedadSuelo': suelo_list,
    'TemperaturaAire': temp_list,
    'HumedadAire': hum_aire_list,
    'TiempoRiego': tiempo_riego_list
})

df.to_csv('dataset_riego_sprint1.csv', index=False)
print("✅ Archivo 'dataset_riego_sprint1.csv' creado exitosamente.")

Generando 20000 datos de entrenamiento...
✅ Archivo 'dataset_riego_sprint1.csv' creado exitosamente.


Entrenamiento

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

# 1. CARGAR DATOS
# Si ya tienes el archivo subido, usa: df = pd.read_csv('dataset_riego_sprint1.csv')
# Si lo generaste en la celda anterior, el dataframe 'df' ya existe en memoria.
# Por seguridad, intentamos leerlo:
try:
    df = pd.read_csv('dataset_riego_sprint1.csv')
    print("✅ Dataset cargado correctamente.")
except:
    print("⚠️ No encontré el archivo CSV. Asegúrate de haber ejecutado el generador primero.")

# Separar Entradas (Features) y Salidas (Labels)
# Entradas: HumedadSuelo, TemperaturaAire, HumedadAire
X = df[['HumedadSuelo', 'TemperaturaAire', 'HumedadAire']]
# Salida: TiempoRiego
y = df['TiempoRiego']

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. CREAR EL MODELO (RED NEURONAL)
model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=[3]), # Capa oculta 1
    keras.layers.Dense(8, activation='relu'),                   # Capa oculta 2
    keras.layers.Dense(1)                                       # Salida única (Tiempo)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

print("🚀 Entrenando modelo (esto tomará unos segundos)...")
history = model.fit(X_train, y_train, epochs=100, validation_split=0.1, verbose=0)
print("✅ Entrenamiento finalizado.")

# 3. CONVERTIR A TENSORFLOW LITE (TinyML)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Optimizaciones para microcontroladores
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Guardar archivo .tflite físico
with open('riego_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ Modelo convertido a TFLite.")

# 4. GENERAR CÓDIGO C++ (HEX DUMP)
# Esta función convierte los bytes del modelo en un array de C
def hex_to_c_array(model_data, model_name="riego_model"):
    c_str = '#include <cstdint>\n\n'
    c_str += f'alignas(8) const unsigned char {model_name}[] = {{\n'
    hex_array = []
    for i, val in enumerate(model_data):
        hex_array.append(f'0x{val:02x}')
    c_str += ', '.join(hex_array)
    c_str += f'\n}};\n\nconst int {model_name}_len = {len(model_data)};'
    return c_str

c_code = hex_to_c_array(tflite_model)

# Guardar el archivo header
with open('modelo_entrenado.h', 'w') as f:
    f.write(c_code)

print("\n" + "="*40)
print("👇 COPIA ESTE CÓDIGO EN TU PROYECTO 👇")
print("="*40)
print(c_code)
print("="*40)
print("💾 También se ha guardado como 'modelo_entrenado.h' en los archivos de Colab.")

✅ Dataset cargado correctamente.
🚀 Entrenando modelo (esto tomará unos segundos)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ Entrenamiento finalizado.
Saved artifact at '/tmp/tmpl7mhlbsf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135788462865296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135788462866256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135788462864144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135788462863568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135788462865104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135788462863760: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Modelo convertido a TFLite.

👇 COPIA ESTE CÓDIGO EN TU PROYECTO 👇
#include <cstdint>

alignas(8) const unsigned char riego_model[] = {
0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33, 0x14, 0x00, 0x20, 0x00, 0x1c, 0x00, 0x18, 0x00, 0x14, 0x00, 0x10, 0x00, 0x0c, 0x00, 0x00, 0x0